<a href="https://colab.research.google.com/github/cyrus1557/investment-readiness-capstone-project/blob/main/index.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![banner image](path/or/url/to/image)
*Add a banner image here (optional).*

# Investment Readiness and Recommendation System
### Predicting who is ready to invest in Kenya, why, and which options suit them (FinAccess 2024)

**Moringa School · Data Science · Module 6 Capstone**
**Team:** [group name] · [member names]
**Technical Mentor:** Samuel

## Overview (write this last)
*Fill this in at the very end, once the results are in. Keep it to one short paragraph:*
- **Bottom line up front:** what did we build and how well does it work (final PR-AUC vs the 0.027 baseline)?
- What are the 2 or 3 biggest things that affect whether a Kenyan adult invests?
- What should the people who'd use this do with it (users, SACCOs and fintechs, financial educators)?

[write here]

## How we work in this notebook
1. **One notebook for the whole team.** Always open `index.ipynb` from our shared Drive folder. Never use *File → Save a copy in Drive*, because that creates duplicates.
2. **Work in your own section.** Your name is in the table below. Don't edit someone else's section without telling them.
3. **The data is read-only.** We only *read* the CSV from Drive. Nothing in this notebook should save files back into the Drive folder.
4. **It must run top to bottom.** Before you finish for the day, do *Runtime → Restart and run all* and make sure nothing breaks.
5. **Use the shared names.** Use the variable names in the table below so everyone's code fits together.

| # | Section | Owner | Status |
|---|---|---|---|
| 1 | Business Understanding | [name] | Done |
| 2 | Data Understanding | [name] | Not started |
| 3 | Exploratory Data Analysis | [name] | Not started |
| 4 | Data Preparation | [name] | Not started |
| 5 | Modelling | [name] | Not started |
| 6 | Evaluation | [name] | Not started |
| 7 | Explainability (SHAP) | [name] | Not started |
| 8 | Recommendation Layer | [name] | Not started |
| 9 | Deployment (Streamlit + LLM) | [name] | Not started |
| 10 | Conclusions | [name] | Not started |

**Shared variable names**

| Name | What it holds |
|---|---|
| `df` | the full dataset as loaded |
| `TARGET`, `WEIGHT` | the target column name and the survey weight column name |
| `FEATURES` | list of the 12 model input columns |
| `OPTIONS` | list of the 6 investment option columns (suggestions only) |
| `EDA_ONLY` | list of columns used for charts only (`Quintiles`) |
| `X`, `y` | model inputs and target (1 = invests, 0 = doesn't) |
| `X_train`, `X_test`, `y_train`, `y_test` | the 80/20 split |
| `preprocessor` | the ColumnTransformer |
| `results` | the model comparison table |
| `final_model` | the chosen pipeline (preprocessing + model) |
| `RANDOM_STATE` | 42, used everywhere |

# 1. Business Understanding

## 1.1 Background
Kenya has a growing range of regulated investment products: Treasury bills and bonds, shares on the Nairobi Securities Exchange, money market funds, unit trusts, SACCOs and pension schemes. Yet formal investing is still rare. In the 2024 FinAccess Household Survey, only **566 of 20,871 adults (about 2.7%)** said they use a formal investment product, even though many more have the income, education and phone access that investing needs.

Most people save through mobile money, savings accounts, chamas or cash at home, and decide where to put money based on word of mouth. Without clear, personal guidance, many capable adults never start, stay in low-return savings, or end up in unsuitable or fraudulent schemes.

## 1.2 Stakeholders
| Stakeholder | What they need |
|---|---|
| **Kenyan adults and first-time investors** | To know how ready they are to invest, what is holding them back, and which safe, regulated options people like them use. |
| **SACCOs, fund managers and fintechs** | To find adults who are close to investing but haven't started, and offer them suitable products. |
| **Financial literacy programmes and regulators** (e.g. CMA, CBK) | To know which factors hold people back most, so they can focus education and policy where it matters. |

## 1.3 Problem Statement
Potential investors in Kenya lack clear, personal guidance on whether they are ready to invest and which options suit their situation. At the same time, financial institutions and literacy programmes can't easily tell which adults are close to investing, or what specifically is stopping them. As a result, outreach is untargeted, and many capable adults remain in low-return or informal savings.

This project addresses the problem by **predicting how ready a person is to invest** from a few simple questions they can answer, **explaining the main factors behind their score**, and **suggesting investment options** that similar people actually use.

## 1.4 Objectives

**Main objective**

To build and deploy an explainable machine learning app that estimates a Kenyan adult's readiness to invest, shows what is helping or holding them back, and suggests suitable investment options, using the 2024 FinAccess Household Survey.

**Specific objectives**
1. Identify the demographic, income, digital access and financial literacy factors linked to formal investing in Kenya.
2. Build baseline models and compare at least three classification models, using **PR-AUC** as the main metric because investors are rare (2.7%).
3. Explain each prediction with **SHAP**, so every user sees the main reasons behind their score.
4. Build a recommendation layer that suggests investment options based on what similar people use.
5. Deploy a **Streamlit** app that returns a readiness score, the main reasons and suggested options, with an **LLM** that explains the results in plain language.

## 1.5 Why classification?
Our target, `Investment_usage_fnl`, is a yes or no answer: does this person invest right now? That makes this a **binary classification** problem.

We don't just want a yes or no, though. The model's **probability** of "yes" becomes the person's **readiness score**: how closely they resemble the people who already invest.

## 1.6 Success Metrics

**What mistakes cost us**
- **False positive** (we say "ready" but they aren't): a wasted marketing or outreach effort, or a person nudged too early.
- **False negative** (we say "not ready" but they are): a missed person who could have started investing.

Missing ready people is the bigger loss for this project, but flagging everyone is useless too. That's why we choose the cut-off carefully (Section 6).

| Metric | Why we use it | Target |
|---|---|---|
| **PR-AUC** (main) | Measures how well the model finds the rare investors. Not fooled by the imbalance. | Clearly above random guessing (**0.027**) and above the logistic regression baseline |
| **Recall and precision** at our chosen cut-off | How many ready people we find, and how many of our "ready" flags are right | Reported and explained |
| **ROC-AUC and F1** (secondary) | Overall ranking ability and balance | Reported for comparison |
| **Leakage check** | Guards against results that are too good to be true | No single column almost perfectly predicts the target |
| **Recommendation hit-rate** | Do our suggestions match what people actually use? | Better than just suggesting the most popular product |

**Why not accuracy?** Only 2.7% of people invest. A model that says "nobody invests" would be 97% accurate and completely useless.

# 2. Data Understanding

## 2.1 Data Source
- **Dataset:** 2024 FinAccess Household Survey, public microdata (Central Bank of Kenya, KNBS and FSD Kenya)
- **Link:** https://finaccess.knbs.or.ke/reports-and-datasets
- **Full survey:** 20,871 adults and 3,816 columns
- **Our extract:** 21 columns, agreed by the team after feedback from our Technical Mentor. *(How we extracted it is described in our data extraction walkthrough document.)*

| Group | Column | Meaning |
|---|---|---|
| **Target** | `Investment_usage_fnl` | Does this person invest right now? ("Usage" / "Non-usage") |
| **Weight** | `hhWeight` | Survey weight: how many Kenyans each person represents. **EDA only.** |
| **Model inputs (12)** | `A18` | Age |
| | `A13` | Sex |
| | `A08` | Urban or rural |
| | `county` | County |
| | `Education` | Highest level of education |
| | `Marital` | Marital status |
| | `NHM` | Household size |
| | `livelihoodcat` | Main source of income / type of work |
| | `B3Ii` | Monthly income (KSh) |
| | `ID_card` | Has a national ID |
| | `mobile_money_access` | Uses mobile money |
| | `fl_score` | Financial literacy quiz (how many of 3 questions answered correctly) |
| **Investment options (6)** | `C1_33` | Shares, bonds, Treasury bills |
| *(suggestions only, NEVER model inputs)* | `C1_34` | Unit trusts, money market funds, REITs |
| | `Sacco_usage` | Uses a SACCO |
| | `Pension_usage` | Has a pension |
| | `chama_usage` | In a chama |
| | `Savings_usage` | Has savings |
| **EDA only** | `Quintiles` | Wealth group (calculated by the survey team). Not a model input, because the app can't ask for it. |

**Why these 12 inputs?** They are things a normal person can answer about themselves on our app form. If the app can't ask it, it can't be a model input.

## 2.2 Setup: Imports
Import everything the notebook needs in this one cell, so it's easy to see what we depend on.

In [ ]:
# TODO: Import the libraries we need
#   - Data:          pandas, numpy
#   - Charts:        matplotlib, seaborn
#   - Modelling:     train_test_split, StratifiedKFold, cross_validate, GridSearchCV / RandomizedSearchCV
#                    ColumnTransformer, Pipeline, SimpleImputer, OneHotEncoder, StandardScaler
#                    DummyClassifier, LogisticRegression, RandomForestClassifier
#                    XGBoost or LightGBM (may need: !pip install xgboost / lightgbm)
#   - Metrics:       average_precision_score, roc_auc_score, precision_recall_curve,
#                    classification_report, confusion_matrix, ConfusionMatrixDisplay
#   - Saving:        joblib
#
# TODO: Set RANDOM_STATE = 42 and use it everywhere
# TODO: (optional) Set display options so we can see all columns

## 2.3 Loading the Data
- Connect Google Drive and read the dataset from our shared folder.
- **Read only.** Don't save anything back to the Drive folder (no `to_csv`, no downloads into Drive). That's what keeps the folder clean.
- If you see *"file not found"*: the folder was shared with you, so in Drive right-click the folder and choose **Organize → Add shortcut** to put it in *My Drive*. That's a shortcut, not a copy.

In [ ]:
# TODO: Mount Google Drive
# TODO: Set the path to our dataset in the shared folder:
#       /content/drive/MyDrive/Capstone_FinAccess/finaccess_2024_capstone.csv
# TODO: Load it into a DataFrame called df
# TODO: (optional) Also load the column guide CSV to look up what each column means

## 2.4 Column Groups
Define the column lists from the table in 2.1, so the rest of the notebook can use them by name.

In [ ]:
# TODO: Define:
#   TARGET   = the target column name
#   WEIGHT   = the survey weight column name
#   FEATURES = list of the 12 model input columns
#   OPTIONS  = list of the 6 investment option columns
#   EDA_ONLY = ["Quintiles"]
#
# TODO: Safety check (assert): every column in df belongs to exactly one group, and
#       none of OPTIONS / WEIGHT / EDA_ONLY are in FEATURES

## 2.5 First Look at the Data

In [ ]:
# TODO: Show the shape, the first few rows, and .info()

In [ ]:
# TODO: How many people invest? Show counts and % of the sample
# TODO: Show the same % using hhWeight (weighted % of Kenyan adults)

In [ ]:
# TODO: Summary statistics for the numeric columns (age, household size, income)

## 2.6 Data Quality
Check the data before changing anything.

In [ ]:
# TODO: % missing in every column (sorted, highest first)
# TODO: Check for duplicate rows

In [ ]:
# TODO: Value counts for every category column
#       Look for odd answers such as "Don't know" or "Refused to answer"

In [ ]:
# TODO: Look closely at income (B3Ii): its distribution, very high values, and how skewed it is
# TODO: Check that age and household size look sensible

### Observations: Data Understanding
*Answer these in a few bullet points:*
1. Which columns have missing values, and what might the gaps mean? (For example, did some people choose not to share their income?)
2. Which columns have odd answers ("Don't know", "Refused") that need a decision?
3. Is income very skewed? What might we do about it (cap the highest values, or use a log)?
4. Which columns are numbers, which are categories, and which are ordered categories (education, literacy score)?

[write here]

# 3. Exploratory Data Analysis
Goal: tell the story of **who invests in Kenya**.

**Rule:** whenever we write "X% of Kenyans", use `hhWeight`. Raw counts are fine for "how many people in our data".

## 3.1 Distributions

In [ ]:
# TODO: Charts of the main features on their own (histograms for numbers, bar charts for categories)

## 3.2 Who Invests? (weighted)
Tip: write one small helper function, e.g. `weighted_rate(df, column)`, that returns the weighted % who invest for each answer in a column. Then reuse it for every feature.

In [ ]:
# TODO: Write the weighted_rate helper function

In [ ]:
# TODO: Weighted investment rate by: sex, age group, urban/rural, county, education,
#       marital status, livelihood, income band, ID, mobile money, financial literacy
# TODO: Bar charts for the clearest differences

## 3.3 Wealth Group (Quintiles), EDA only

In [ ]:
# TODO: Weighted investment rate by wealth quintile, with a bar chart
# Reminder: Quintiles is NOT a model input

## 3.4 Investment Options

In [ ]:
# TODO: Among the 566 investors: how many hold shares/bonds (C1_33) vs funds (C1_34)?
# TODO: Across everyone: how many use a SACCO, pension, chama or savings?
# TODO: Chart the results (this feeds the recommendation layer in Section 8)

## 3.5 Relationships Between Features

In [ ]:
# TODO: How do the features relate to each other? (e.g. income vs education, age vs income)
#       Use crosstabs or a correlation heatmap for the numeric columns

### Key Insights from the EDA
*Write the 3 to 5 most interesting findings in plain English. These become slides.*
1. [write here]
2. [write here]
3. [write here]

# 4. Data Preparation
Goal: turn the raw answers into something a model can learn from, **without leaking the answer**.

Write down every decision and why. We need these for the report.

## 4.1 Target and Inputs

In [ ]:
# TODO: Create y: 1 if the target is "Usage", 0 otherwise
# TODO: Create X from FEATURES only (NOT OPTIONS, WEIGHT or Quintiles)

## 4.2 Leakage Check
A column "leaks" if it gives away the answer. A leaking model looks amazing in testing and is useless in real life.

In [ ]:
# TODO: For each input column, check the investor rate for each answer
#       If any answer (with a decent number of people) has a very high investor rate, investigate it
# TODO: Confirm none of OPTIONS are in X

## 4.3 Cleaning Decisions

In [ ]:
# TODO: Literacy (fl_score): convert the words to a number from 0 to 3
#       ("None correct" = 0 ... "All correct" = 3)
# TODO: Decide what to do with very high incomes (cap or log) and apply it
# TODO: Decide what to do with "Don't know" / "Refused" answers (group them, or treat as missing)
# NOTE: Fill missing values INSIDE the pipeline (4.6), not here, so the test set stays untouched

## 4.4 Feature Engineering

In [ ]:
# TODO: (optional) New features to try, e.g. age bands, income per household member
#       Only keep them if they help the model

## 4.5 Train/Test Split

In [ ]:
# TODO: Split X and y into 80% train and 20% test
#       stratify=y (so both parts keep the 2.7% investor rate), random_state=RANDOM_STATE
# TODO: Print the size and number of investors in each part
# IMPORTANT: Don't touch X_test / y_test again until Section 6.1

## 4.6 Preprocessing Pipeline
Must use `ColumnTransformer` and `Pipeline`, so preprocessing only ever learns from the training data.

In [ ]:
# TODO: Make two lists: numeric columns and category columns
# TODO: Numeric pipeline: fill missing with the median (add_indicator=True flags missing income), then scale
# TODO: Category pipeline: fill missing with the most common answer, then one-hot encode
#       (handle_unknown="ignore")
# TODO: Combine them in a ColumnTransformer called preprocessor

### Notes: Data Preparation decisions
| Decision | What we did | Why |
|---|---|---|
| Missing income | [ ] | [ ] |
| High incomes | [ ] | [ ] |
| "Don't know" / "Refused" | [ ] | [ ] |
| Literacy score | Converted to 0 to 3 | It's an ordered score |
| Scaling | [ ] | [ ] |

# 5. Modelling
Goal: start simple, then improve. Every model goes in one pipeline (`preprocessor` + model) and is judged with **5-fold stratified cross-validation on the training data**, using **PR-AUC**.

## 5.1 Validation Strategy

In [ ]:
# TODO: Set up StratifiedKFold (5 folds, shuffle=True, random_state=RANDOM_STATE)
# TODO: Scoring: "average_precision" (PR-AUC) and "roc_auc"
# TODO: Create an empty results list/table, and a small helper that cross-validates a pipeline
#       and adds its mean PR-AUC, its spread and ROC-AUC to the table

## 5.2 Baseline Models
1. **Dummy model.** It guesses without looking at anyone. Its PR-AUC should be about 0.027. Every real model must beat this.
2. **Logistic regression** with `class_weight="balanced"`. Simple and easy to explain, and the model to beat.

In [ ]:
# TODO: Baseline 1: DummyClassifier

In [ ]:
# TODO: Baseline 2: LogisticRegression(class_weight="balanced", max_iter=...)

## 5.3 Stronger Models

In [ ]:
# TODO: Random Forest with class_weight="balanced"

In [ ]:
# TODO: XGBoost or LightGBM. For the imbalance use scale_pos_weight (about non-investors / investors)

## 5.4 Hyperparameter Tuning
Tune the best one or two models. Use `scoring="average_precision"`.

*Be mindful of time: show **how** we tuned. We don't need to search everything.*

In [ ]:
# TODO: Define a parameter grid (e.g. max_depth, n_estimators, learning_rate, C for logistic regression)
# TODO: Run GridSearchCV or RandomizedSearchCV on the pipeline (training data only)
# TODO: Show the best parameters and best cross-validated PR-AUC, and add it to the results table

## 5.5 Model Comparison

In [ ]:
# TODO: Show the results table sorted by PR-AUC
# TODO: Bar chart of PR-AUC per model, with a line at 0.027 (random guessing)

### Observations: Modelling
1. Which model did best on PR-AUC, and by how much over the baselines?
2. Are the scores stable across folds (small spread)?
3. **Plan B:** if no model is clearly above the baselines, consider adding household questions (the ingredients of the wealth quintile) and test again.

[write here]

# 6. Evaluation
Choose the final model, then test it **once** on the test set we locked away in 4.5.

## 6.1 Final Model on the Test Set

In [ ]:
# TODO: Fit the chosen pipeline on ALL of the training data, and call it final_model
# TODO: Predict probabilities for X_test
# TODO: Report PR-AUC and ROC-AUC on the test set
# TODO: Plot the precision-recall curve, with the 0.027 baseline line

## 6.2 Choosing the Cut-off (Threshold)
The default cut-off of 0.5 rarely works when only 2.7% are investors. Pick one that balances finding ready people against flagging too many.

In [ ]:
# TODO: Show precision and recall at different cut-offs
# TODO: Choose a cut-off and explain why
# TODO: Confusion matrix and classification report at that cut-off

## 6.3 Fairness and Segment Check

In [ ]:
# TODO: Does the model work about as well for different groups?
#       e.g. men vs women, urban vs rural, young vs older

## 6.4 From Probability to Readiness Score
Raw probabilities will be low for most people (often under 10%), which looks broken in an app. Turn them into something people understand.

In [ ]:
# TODO: Convert probabilities to a percentile ("more ready than X% of Kenyan adults")
#       or to bands (Low / Medium / High). Decide which with the team.

### Observations: Evaluation
1. How did the final model do on the test set compared with cross-validation?
2. Which cut-off did we choose, and what does it mean in practice?
3. Any groups where the model works noticeably worse?

[write here]

# 7. Explainability (SHAP)
Goal: explain what pushes a person's readiness score up or down, overall and for individual people.

In [ ]:
# TODO: !pip install shap (if needed) and import it
# TODO: Global importance: which features matter most overall? (summary / bar plot)

In [ ]:
# TODO: Dependence plots for the top 2 or 3 features (e.g. how income affects the score)

In [ ]:
# TODO: Pick 2 or 3 example people and show why each got their score (waterfall plot)
# TODO: Turn the top reasons into plain sentences for the app,
#       e.g. "Your income is the biggest factor holding your score down"

### Observations: Explainability
1. What are the top factors behind investment readiness?
2. Which of them can a person actually change (e.g. financial literacy, mobile money), and which can't they (age, county)?
3. What does this suggest for SACCOs, fintechs and financial educators?

[write here]

# 8. Recommendation Layer
Goal: for a new person, suggest investment options based on what **similar people** use.

## 8.1 Product Reference Table
*Complete this table with the team. Research the details from official sources.*

| Option | Matching column | Risk | Typical minimum | How easy to withdraw | Time horizon | Regulator |
|---|---|---|---|---|---|---|
| Treasury bills and bonds | `C1_33` | [ ] | [ ] | [ ] | [ ] | CBK |
| Shares (NSE) | `C1_33` | [ ] | [ ] | [ ] | [ ] | CMA |
| Money market funds and unit trusts | `C1_34` | [ ] | [ ] | [ ] | [ ] | CMA |
| SACCO savings and shares | `Sacco_usage` | [ ] | [ ] | [ ] | [ ] | SASRA |
| Pension scheme | `Pension_usage` | [ ] | [ ] | [ ] | [ ] | RBA |
| Chama | `chama_usage` | [ ] | [ ] | [ ] | [ ] | [ ] |

## 8.2 Finding Similar People

In [ ]:
# TODO: Use nearest neighbours on the (preprocessed) model inputs to find people most like a given user
# TODO: Look at what those similar people use from OPTIONS
# NOTE: Funds (C1_34) only have about 31 users. Use everyone who uses a SACCO, pension or chama,
#       not just the 566 investors, so suggestions are well supported

## 8.3 The Recommend Function

In [ ]:
# TODO: Write recommend(profile) that returns the top options for one person,
#       ranked by how common they are among similar people, with details from the 8.1 table

## 8.4 Checking the Recommendations

In [ ]:
# TODO: On the test set, how often do our top suggestions match what people actually use?
# TODO: Compare with a simple baseline: always suggest the most popular option

# 9. Deployment (Streamlit + LLM)

## 9.1 Saving the Model
- Save `final_model` (and anything the recommender needs) with `joblib`.
- Save to Colab's local storage (`/content/`) and download it, or commit it to **GitHub**. **Don't save model files into the shared Drive folder.**

In [ ]:
# TODO: Save the final pipeline and the recommender files with joblib

## 9.2 Streamlit App
The app lives in our GitHub repo (`app.py`), not in this notebook.

**The form asks:** age, sex, county, urban/rural, household size, education, marital status, type of work, monthly income, has an ID, uses mobile money, and a **3-question literacy quiz** (interest, inflation, spreading risk).

**The app shows:**
1. the readiness score (percentile or band from 6.4)
2. the top reasons (from SHAP)
3. suggested options (from Section 8)
4. a note: *"For learning, not financial advice."*

Deploy on Streamlit Community Cloud and add the link here: [link]

## 9.3 LLM Layer
The LLM **explains, it does not decide.**
- **Input to the LLM:** the readiness score, top reasons and suggested options from our model, plus the product table from 8.1.
- **Rules in the prompt:** only mention products from our table; plain, friendly language; never suggest unregulated schemes or gambling; always end with the disclaimer.
- **Follow-up questions:** let users ask things like "What is a money market fund?"
- **API key:** keep it in Streamlit secrets. Never put it in this notebook or on GitHub.

In [ ]:
# TODO: (optional) Test the LLM prompt here with one example person before adding it to the app

## 9.4 End-to-End Test

In [ ]:
# TODO: Run 3 example people through the whole flow (score, reasons, suggestions)
#       e.g. a young urban graduate, a rural farmer, a middle-aged business owner

# 10. Conclusions

## 10.1 Findings
- [write here]

## 10.2 Model Performance
- Final model: [ ] · Test PR-AUC: [ ] (random guessing: 0.027) · Recall: [ ] · Precision: [ ]

## 10.3 Recommendations
- **For individuals:** [ ]
- **For SACCOs, fintechs and fund managers:** [ ]
- **For financial educators and regulators:** [ ]

## 10.4 Limitations
- Only 2.7% of people invest, and only about 31 hold funds, so the model has few examples to learn from.
- Survey answers are self-reported and from one year (2024). The model finds links, not causes.
- Income is missing for about 11% of people and may be under-reported.
- Suggestions reflect what similar people use, not expected returns or personal suitability.
- [add more]

## 10.5 Next Steps
- [write here]